# CSC51 Multi-Agent Weekly Teaching Pack Generator
## Clean Interactive Capstone Notebook

This version removes the duplicated cells and keeps one clear workflow.

**Architecture**

Teacher selects week → Retrieval Agent → Planning Agent → Grounding Safeguard → Resource Agent → Reviewer Agent → Teaching Pack

**Course techniques used**
- Multi-agent handoffs with LangGraph
- Function calling for curriculum tools
- The same curriculum tools are exposed through an MCP server
- Retrieval from the official CSC51 course specification
- Deterministic input/output safeguards
- LLM-as-a-Judge style final review
- Word, PowerPoint, worksheet, and JSON output generation

> Run the notebook from top to bottom. After a kernel restart, use **Run All**.


## 0. Install packages once

Run this only if your environment is missing packages. Restart the kernel after installation.


In [ ]:
# Uncomment and run only when needed:
# %pip install "mcp<2" langchain langgraph langchain-ollama python-docx python-pptx ipywidgets pypdf pymupdf

## 1. Imports and project settings


In [ ]:
from typing import TypedDict, Any
from pathlib import Path

from docx import Document

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR

from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END

from IPython.display import display
import ipywidgets as widgets

import importlib
import json
import re
import sys

PROJECT_DIR = Path.cwd()
KNOWLEDGE_DIR = PROJECT_DIR / "knowledge"
TEMPLATE_DIR = PROJECT_DIR / "templates"
OUTPUT_DIR = PROJECT_DIR / "outputs"
IMAGE_CACHE_DIR = OUTPUT_DIR / "tutorial_page_images"

COURSE_SPEC_PATH = KNOWLEDGE_DIR / "AY26-27-CSC51-Course specifications.docx"
PYTHON_TUTORIAL_PATH = KNOWLEDGE_DIR / "PythonTutorial.pdf"
LESSON_TEMPLATE_PATH = TEMPLATE_DIR / "Approved GRM - Template - Shared.docx"

MODEL_NAME = "qwen3:4b"
GRADE = "Grade 10 ADV"
COURSE = "CSC51 Applied Programming"

OUTPUT_DIR.mkdir(exist_ok=True)
IMAGE_CACHE_DIR.mkdir(exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Course specification found:", COURSE_SPEC_PATH.exists())
print("Python tutorial found:", PYTHON_TUTORIAL_PATH.exists())
print("Lesson template found:", LESSON_TEMPLATE_PATH.exists())

if not COURSE_SPEC_PATH.exists():
    raise FileNotFoundError(
        "Put 'AY26-27-CSC51-Course specifications.docx' inside the knowledge folder."
    )

if not PYTHON_TUTORIAL_PATH.exists():
    raise FileNotFoundError(
        "Put 'PythonTutorial.pdf' inside the knowledge folder."
    )

if not LESSON_TEMPLATE_PATH.exists():
    raise FileNotFoundError(
        "Put 'Approved GRM - Template - Shared.docx' inside the templates folder."
    )

## 2. Teacher control panel

Choose the week from the dropdown.  
You do **not** need a separate `WEEK = ...` cell.  
The Run cell later reads the dropdown directly.


In [ ]:
week_selector = widgets.Dropdown(
    options=[(f"Week {i}", i) for i in range(1, 12)],
    value=1,
    description="Select Week:"
)

display(week_selector)


## 3. Local Ollama model

`reasoning=False` keeps Qwen3 from spending a long time on hidden reasoning for this classroom generation task.


In [ ]:
# ============================================================
# LOCAL OLLAMA MODELS
# ============================================================
# llm: lightweight general model
# json_llm: planning/reviewer JSON
# resource_llm: larger output budget for rich PowerPoint + worksheet JSON

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
    reasoning=False,
    num_predict=700
)

json_llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
    reasoning=False,
    format="json",
    num_predict=2400
)

resource_llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
    reasoning=False,
    format="json",
    num_predict=5000
)

print("Ollama models ready:", MODEL_NAME)

## 4. Single curriculum module

This cell creates **one source of truth** for:
- the 22-period semester sequence
- the 11-week map
- official PC retrieval
- resource retrieval
- Word-table extraction

Both the notebook and the MCP server use this same module, so the curriculum logic is not duplicated.


In [ ]:
# ============================================================
# STEP 4 - SINGLE CURRICULUM + PYTHON TUTORIAL RETRIEVAL MODULE
# ============================================================

curriculum_module_code = r"""
from pathlib import Path
from docx import Document
from pypdf import PdfReader
import re

PROJECT_DIR = Path(__file__).resolve().parent

COURSE_SPEC_PATH = (
    PROJECT_DIR
    / "knowledge"
    / "AY26-27-CSC51-Course specifications.docx"
)

PYTHON_TUTORIAL_PATH = (
    PROJECT_DIR
    / "knowledge"
    / "PythonTutorial.pdf"
)


# ============================================================
# OFFICIAL SEMESTER SEQUENCE
# ============================================================
# 22 periods = 11 weeks x 2 periods

SEMESTER_PERIODS = [
    "List: collection of data",
    "Sorting list",
    "Sorting list",
    "Processing list",
    "Processing list",
    "Multidimensional array",
    "Multidimensional array",

    "Mutability and sequence",
    "Mutable and immutable data types",
    "Tuple",
    "Tuple",
    "Dictionary",
    "Dictionary",

    "Robotic System: Hardware & Software",
    "MicroPython Programming",
    "AIoT Controllers",
    "AIoT Controllers",
    "Sensors and Data",
    "Sensors and Data",
    "Applications using AI Camera",
    "Applications using AI Camera",
    "Applications using AI Camera",
]


# ============================================================
# WORD TABLE EXTRACTION
# ============================================================

def extract_table_rows(table):
    rows = []

    for row in table.rows:
        values = []

        for cell in row.cells:
            text = cell.text.strip()

            if text:
                values.append(text)

            for nested in cell.tables:
                rows.extend(
                    extract_table_rows(nested)
                )

        if values:
            rows.append(" | ".join(values))

    return rows


def load_course_rows():
    if not COURSE_SPEC_PATH.exists():
        raise FileNotFoundError(
            f"Course specification not found: {COURSE_SPEC_PATH}"
        )

    document = Document(COURSE_SPEC_PATH)

    rows = []

    for table in document.tables:
        rows.extend(
            extract_table_rows(table)
        )

    return rows


# ============================================================
# 11-WEEK MAP
# ============================================================

def build_semester_weeks():
    if len(SEMESTER_PERIODS) != 22:
        raise ValueError(
            "Semester must contain exactly 22 periods."
        )

    return {
        week: {
            "period_1": SEMESTER_PERIODS[(week - 1) * 2],
            "period_2": SEMESTER_PERIODS[(week - 1) * 2 + 1],
        }
        for week in range(1, 12)
    }


def get_week_data(week: int):
    if week < 1 or week > 11:
        raise ValueError(
            "Week must be between 1 and 11."
        )

    return build_semester_weeks()[week]


# ============================================================
# PRECISE PC MAPPING
# ============================================================

def pc_ids_for_topic(topic: str):
    t = topic.strip().lower()

    # Topic 1 - Lists
    if "collection of data" in t:
        return ["PC1.1", "PC1.2"]

    if "sorting" in t:
        return ["PC1.3"]

    if "processing" in t:
        return [
            "PC1.4",
            "PC1.5",
            "PC1.6",
            "PC1.7",
            "PC1.8",
            "PC1.9",
        ]

    if "multidimensional" in t:
        return ["PC1.7", "PC1.10"]

    # Topic 2 - Tuples / dictionaries
    if "mutability and sequence" in t:
        return ["PC2.1", "PC2.2"]

    if "mutable and immutable" in t:
        return ["PC2.1", "PC2.2"]

    if "tuple" in t:
        return [
            "PC2.1",
            "PC2.2",
            "PC2.3",
            "PC2.4",
            "PC2.5",
        ]

    if "dictionary" in t:
        return [
            "PC2.6",
            "PC2.7",
            "PC2.8",
        ]

    # Topic 3 - Robotics / AIoT
    if "robotic system" in t:
        return ["PCA.1"]

    if "micropython" in t:
        return ["PCA.2"]

    if "aiot controllers" in t:
        return ["PCA.3"]

    if "sensors and data" in t:
        return ["PCA.4", "PCA.5"]

    if "ai camera" in t:
        return [
            "PCA.6",
            "PCA.7",
            "PCA.8",
            "PCA.9",
            "PCA.10",
        ]

    return []


def get_performance_criteria(topic: str):
    rows = load_course_rows()
    pc_ids = pc_ids_for_topic(topic)

    if not pc_ids:
        return (
            "No matching performance criteria "
            f"were found for topic: {topic}"
        )

    selected = []

    for row in rows:
        first_field = row.split("|")[0].strip()

        if first_field in pc_ids:
            selected.append(row)

    selected = list(dict.fromkeys(selected))

    if not selected:
        return (
            "No matching performance criteria "
            f"were found for topic: {topic}"
        )

    return "\n".join(selected)


# ============================================================
# TRUSTED RESOURCES
# ============================================================

def get_resources(topic: str):
    t = topic.strip().lower()

    python_essentials_link = (
        "https://www.netacad.com/courses/"
        "python-essentials-1"
        "?courseLang=en-US"
        "&instance_id="
        "c67d18ac-fe42-4e41-a337-09c1dbf540a9"
    )

    if "collection of data" in t:
        resources = [
            "Python Essentials 1 - Cisco Networking Academy",
            f"Course link: {python_essentials_link}",
            "Recommended lesson area: Module 3, Section 3.4 - List",
            "Local knowledge source: PythonTutorial.pdf",
            "Python Videos Folder",
        ]

    elif "sorting" in t:
        resources = [
            "Python Essentials 1 - Cisco Networking Academy",
            f"Course link: {python_essentials_link}",
            "Recommended lesson area: Module 3, Section 3.5 - Sorting Simple List",
            "Local knowledge source: PythonTutorial.pdf",
            "Python Videos Folder",
        ]

    elif "processing" in t:
        resources = [
            "Python Essentials 1 - Cisco Networking Academy",
            f"Course link: {python_essentials_link}",
            "Recommended lesson area: Module 3, Section 3.6 - List Processing",
            "Local knowledge source: PythonTutorial.pdf",
            "Python Videos Folder",
        ]

    elif "multidimensional" in t:
        resources = [
            "Python Essentials 1 - Cisco Networking Academy",
            f"Course link: {python_essentials_link}",
            "Recommended lesson area: Module 3, Section 3.7 - Multidimensional Arrays",
            "Local knowledge source: PythonTutorial.pdf",
            "Python Videos Folder",
        ]

    elif any(
        x in t
        for x in [
            "tuple",
            "mutability",
            "mutable",
            "immutable",
            "dictionary",
        ]
    ):
        resources = [
            "Python Essentials 1 - Cisco Networking Academy",
            f"Course link: {python_essentials_link}",
            "Recommended lesson area: Module 4, Sections 4.1-4.5",
            "Local knowledge source: PythonTutorial.pdf",
            "Python Videos Folder",
        ]

    else:
        resources = [
            "Maker and Coder Portal - MC 4.0 Curriculum",
            "Maker and Coder Portal: https://learningportal.makerandcoder.com/course/us",
            "MC4.0 AIOT Kit",
            "MC Lab Software",
            "Grade 10 Lessons 1-9",
        ]

    return "\n".join(resources)


# ============================================================
# LOCAL PYTHON TUTORIAL PDF RETRIEVAL
# ============================================================

def load_python_tutorial_pages():
    if not PYTHON_TUTORIAL_PATH.exists():
        raise FileNotFoundError(
            f"Python tutorial not found: {PYTHON_TUTORIAL_PATH}"
        )

    reader = PdfReader(
        str(PYTHON_TUTORIAL_PATH)
    )

    pages = []

    for page_number, page in enumerate(
        reader.pages,
        start=1
    ):
        text = page.extract_text() or ""

        text = re.sub(
            r"\s+",
            " ",
            text
        ).strip()

        if text:
            pages.append({
                "page": page_number,
                "text": text,
            })

    return pages


def tutorial_keywords_for_topic(topic: str):
    t = topic.strip().lower()

    if "collection of data" in t:
        return [
            "list",
            "lists",
            "append",
            "insert",
            "index",
            "indices",
            "nested list",
        ]

    if "sorting" in t:
        return [
            "sort",
            "sorted",
            "sorting",
            "reverse",
            "bubble sort",
            "len",
        ]

    if "processing" in t:
        return [
            "slicing",
            "slice",
            "del",
            "in operator",
            "not in",
            "list comprehension",
            "loop",
        ]

    if "multidimensional" in t:
        return [
            "nested list",
            "multidimensional",
            "two dimensional",
            "2d",
            "matrix",
        ]

    if any(
        x in t
        for x in [
            "mutability",
            "mutable",
            "immutable",
            "tuple",
        ]
    ):
        return [
            "tuple",
            "immutable",
            "sequence",
            "mutable",
        ]

    if "dictionary" in t:
        return [
            "dictionary",
            "dict",
            "key",
            "value",
            "items",
        ]

    return []


def get_tutorial_material(
    topic: str,
    max_pages: int = 4,
    max_chars_per_page: int = 2600
):
    # Retrieve the most relevant text pages from PythonTutorial.pdf.
    # PAGE markers are preserved so the Resource Agent can cite the
    # source and Step 12 can render that PDF page as a slide visual.

    keywords = tutorial_keywords_for_topic(topic)

    if not keywords:
        return (
            "No PythonTutorial.pdf retrieval is configured "
            f"for this non-Python topic: {topic}"
        )

    pages = load_python_tutorial_pages()

    scored = []

    for item in pages:
        lower_text = item["text"].lower()

        score = 0

        for keyword in keywords:
            keyword_lower = keyword.lower()

            # Phrase hits get a little extra weight.
            hits = lower_text.count(keyword_lower)
            score += hits * (
                3 if " " in keyword_lower else 1
            )

        if score > 0:
            scored.append({
                "score": score,
                "page": item["page"],
                "text": item["text"],
            })

    scored.sort(
        key=lambda x: (
            -x["score"],
            x["page"]
        )
    )

    selected = scored[:max_pages]

    if not selected:
        return (
            "No relevant material was found in "
            f"PythonTutorial.pdf for topic: {topic}"
        )

    chunks = []

    for item in selected:
        excerpt = item["text"][:max_chars_per_page]

        chunks.append(
            "SOURCE: PythonTutorial.pdf\n"
            f"PAGE: {item['page']}\n"
            f"RELEVANCE SCORE: {item['score']}\n"
            f"CONTENT:\n{excerpt}"
        )

    return "\n\n---\n\n".join(chunks)


def get_tutorial_pages_for_topic(topic: str):
    # Return only the best matching page numbers.
    # Useful for diagnostics and PowerPoint source visuals.

    material = get_tutorial_material(topic)

    return [
        int(x)
        for x in re.findall(
            r"PAGE:\s*(\d+)",
            material
        )
    ]
"""

curriculum_module_path = (
    PROJECT_DIR
    / "curriculum_data.py"
)

curriculum_module_path.write_text(
    curriculum_module_code,
    encoding="utf-8"
)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_DIR)
    )

import curriculum_data

importlib.reload(
    curriculum_data
)

semester_weeks = (
    curriculum_data
    .build_semester_weeks()
)

# Quick verification
week1 = curriculum_data.get_week_data(1)

print(" Curriculum + PDF retrieval module ready.")
print(" Python tutorial:", PYTHON_TUTORIAL_PATH.name)
print(
    " Week 1 Period 1 PCs:",
    curriculum_data.pc_ids_for_topic(
        week1["period_1"]
    )
)
print(
    " Week 1 Period 2 PCs:",
    curriculum_data.pc_ids_for_topic(
        week1["period_2"]
    )
)
print(
    " Week 1 Period 1 tutorial pages:",
    curriculum_data.get_tutorial_pages_for_topic(
        week1["period_1"]
    )
)
print(
    " Week 1 Period 2 tutorial pages:",
    curriculum_data.get_tutorial_pages_for_topic(
        week1["period_2"]
    )
)

for week, data in semester_weeks.items():
    print(
        f"Week {week}: "
        f"{data['period_1']} | "
        f"{data['period_2']}"
    )

### Trusted Python resources

The system now uses **two grounded Python sources**:

1. The official CSC51 course specification selects the week, topic and performance criteria.
2. `knowledge/PythonTutorial.pdf` is searched locally for lesson-specific explanations and examples.

The Resource Agent receives the most relevant tutorial excerpts with page numbers. The PowerPoint generator can also render those retrieved PDF pages as source visuals.

The Cisco Networking Academy **Python Essentials 1** link remains a trusted course resource, but the notebook does not scrape password-protected NetAcad pages.

## 5. Function-calling tools

The Retrieval Agent calls these tools.  
They use the official course specification through `curriculum_data.py`.


In [ ]:
@tool
def get_week_topics(week: int) -> str:
    """Return the two CSC51 topics scheduled for a week from 1 to 11."""
    data = curriculum_data.get_week_data(week)
    return json.dumps(data)


@tool
def get_performance_criteria_for_topic(topic: str) -> str:
    """Return official CSC51 performance criteria relevant to a topic."""
    return curriculum_data.get_performance_criteria(topic)


@tool
def get_resources_for_topic(topic: str) -> str:
    """Return course-specified resources relevant to a topic."""
    return curriculum_data.get_resources(topic)


@tool
def get_tutorial_material_for_topic(topic: str) -> str:
    """Retrieve relevant grounded material from local PythonTutorial.pdf."""
    return curriculum_data.get_tutorial_material(topic)


curriculum_tools = [
    get_week_topics,
    get_performance_criteria_for_topic,
    get_resources_for_topic,
    get_tutorial_material_for_topic,
]

print("Function-calling tools ready:")
for t in curriculum_tools:
    print("-", t.name)

## 6. MCP server

This creates `mcp_server.py`.  
It exposes the **same three curriculum operations** through MCP without duplicating the curriculum data.


In [ ]:
mcp_server_code = r"""
from mcp.server.fastmcp import FastMCP
import curriculum_data

mcp = FastMCP("csc51-curriculum")


@mcp.tool()
def get_week_topics(week: int) -> str:
    data = curriculum_data.get_week_data(week)

    return (
        f"Period 1: {data['period_1']}\n"
        f"Period 2: {data['period_2']}"
    )


@mcp.tool()
def get_performance_criteria_for_topic(topic: str) -> str:
    return curriculum_data.get_performance_criteria(topic)


@mcp.tool()
def get_resources_for_topic(topic: str) -> str:
    return curriculum_data.get_resources(topic)


@mcp.tool()
def get_tutorial_material_for_topic(topic: str) -> str:
    return curriculum_data.get_tutorial_material(topic)


if __name__ == "__main__":
    mcp.run(transport="stdio")
"""

mcp_server_path = (
    PROJECT_DIR
    / "mcp_server.py"
)

mcp_server_path.write_text(
    mcp_server_code,
    encoding="utf-8"
)

print("MCP server created:", mcp_server_path)
print(
    "The same curriculum + PythonTutorial.pdf retrieval operations "
    "are exposed through MCP."
)
print(
    "The notebook uses deterministic direct retrieval for reliability "
    "on Windows/Jupyter."
)

## 7. Shared state and JSON helper


In [ ]:
class TeachingState(TypedDict, total=False):
    week: int
    grade: str
    course: str

    weekly_topics: str
    performance_criteria: str
    resources: str
    tutorial_material: str
    curriculum_context: str

    weekly_lesson_plan: dict[str, Any]
    grounding_check: dict[str, Any]

    powerpoint_content: dict[str, Any]
    worksheet: dict[str, Any]

    review: dict[str, Any]
    output_files: list[str]


def parse_json(text):
    """
    Robustly parse a JSON object returned by a local LLM.
    """

    if text is None:
        raise ValueError("Empty model response.")

    if isinstance(text, (dict, list)):
        if isinstance(text, dict):
            return text

        raise ValueError(
            "Expected a JSON object, but received a list."
        )

    text = str(text).strip()

    if not text:
        raise ValueError("Empty model response.")

    text = re.sub(
        r"^\s*```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```\s*$",
        "",
        text
    )

    first = text.find("{")
    last = text.rfind("}")

    if (
        first == -1
        or last == -1
        or last < first
    ):
        raise ValueError(
            "No complete JSON object was found "
            "in the model response."
        )

    json_text = text[
        first:last + 1
    ]

    try:
        return json.loads(
            json_text
        )

    except json.JSONDecodeError as e:
        raise ValueError(
            "Invalid or truncated JSON "
            f"returned by the model: {e}"
        ) from e

## 8. Specialized agents

This clean version uses four specialized agents:
1. Retrieval Agent
2. Planning Agent
3. Resource Agent
4. Reviewer Agent

That reduces unnecessary model calls while still demonstrating a real multi-agent handoff system.


In [ ]:
# ============================================================
# STEP 8 - SPECIALIZED AGENTS
# ============================================================


# ============================================================
# HELPER - STRICT JSON PARSING
# ============================================================

def parse_agent_json(
    response_text,
    agent_name
):
    print(
        f"\n--- {agent_name} output ---"
    )

    preview = str(
        response_text
    )

    if len(preview) > 2500:
        preview = (
            preview[:2500]
            + "\n... output shortened ..."
        )

    print(
        preview
    )

    try:
        data = parse_json(
            response_text
        )

        if not isinstance(
            data,
            dict
        ):
            raise ValueError(
                "Agent output must be a JSON object."
            )

        print(
            f"{agent_name} JSON parsed successfully."
        )

        return data

    except Exception as e:
        print(
            f"\n{agent_name} JSON parsing failed."
        )

        print(
            "Error:",
            e
        )

        raise


# ============================================================
# 1. RETRIEVAL AGENT
# ============================================================

async def retrieval_node(
    state: TeachingState
):
    week = state[
        "week"
    ]

    week_data = (
        curriculum_data
        .get_week_data(
            week
        )
    )

    topic_1 = (
        week_data[
            "period_1"
        ]
    )

    topic_2 = (
        week_data[
            "period_2"
        ]
    )

    pcs_1 = (
        curriculum_data
        .get_performance_criteria(
            topic_1
        )
    )

    pcs_2 = (
        curriculum_data
        .get_performance_criteria(
            topic_2
        )
    )

    official_pc_ids_1 = (
        curriculum_data
        .pc_ids_for_topic(
            topic_1
        )
    )

    official_pc_ids_2 = (
        curriculum_data
        .pc_ids_for_topic(
            topic_2
        )
    )

    resources_1 = (
        curriculum_data
        .get_resources(
            topic_1
        )
    )

    resources_2 = (
        curriculum_data
        .get_resources(
            topic_2
        )
    )

    tutorial_1 = (
        curriculum_data
        .get_tutorial_material(
            topic_1
        )
    )

    tutorial_2 = (
        curriculum_data
        .get_tutorial_material(
            topic_2
        )
    )

    weekly_topics = (
        f"Period 1: {topic_1}\n"
        f"Period 2: {topic_2}"
    )

    performance_criteria = f"""
PERIOD 1
Topic: {topic_1}

Allowed PC IDs:
{official_pc_ids_1}

Official Performance Criteria:
{pcs_1}


PERIOD 2
Topic: {topic_2}

Allowed PC IDs:
{official_pc_ids_2}

Official Performance Criteria:
{pcs_2}
""".strip()

    resources = f"""
PERIOD 1
Topic: {topic_1}

{resources_1}


PERIOD 2
Topic: {topic_2}

{resources_2}
""".strip()

    tutorial_material = f"""
================================
PYTHON TUTORIAL MATERIAL - PERIOD 1
================================

Topic:
{topic_1}

{tutorial_1}


================================
PYTHON TUTORIAL MATERIAL - PERIOD 2
================================

Topic:
{topic_2}

{tutorial_2}
""".strip()

    curriculum_context = f"""
COURSE:
{state["course"]}

GRADE:
{state["grade"]}

WEEK:
{week}


================================
OFFICIAL PERIOD 1
================================

Topic:
{topic_1}

Allowed PC IDs:
{official_pc_ids_1}

Official Performance Criteria:
{pcs_1}


================================
OFFICIAL PERIOD 2
================================

Topic:
{topic_2}

Allowed PC IDs:
{official_pc_ids_2}

Official Performance Criteria:
{pcs_2}


================================
TRUSTED COURSE RESOURCES
================================

{resources}


STRICT CURRICULUM RULES:

1. Period 1 topic MUST remain exactly:
   {topic_1}

2. Period 2 topic MUST remain exactly:
   {topic_2}

3. Period 1 may use ONLY:
   {official_pc_ids_1}

4. Period 2 may use ONLY:
   {official_pc_ids_2}

5. Never invent a topic.

6. Never invent a PC code.

7. PythonTutorial.pdf is a supporting teaching source only.
   It must not choose or replace the official topic or PCs.
""".strip()

    print(
        "\nRetrieval Agent finished."
    )

    print(
        "Retrieved topics:\n",
        weekly_topics
    )

    print(
        "\nOfficial PCs:"
    )

    print(
        "Period 1:",
        official_pc_ids_1
    )

    print(
        "Period 2:",
        official_pc_ids_2
    )

    print(
        "Official performance criteria retrieved."
    )

    print(
        "Trusted lesson resources retrieved."
    )

    print(
        "PythonTutorial.pdf material retrieved for Resource Agent only."
    )

    return {
        "weekly_topics":
            weekly_topics,

        "performance_criteria":
            performance_criteria,

        "resources":
            resources,

        "tutorial_material":
            tutorial_material,

        "curriculum_context":
            curriculum_context,
    }


# ============================================================
# 2. PLANNING AGENT
# ============================================================

def plan_one_period(
    state,
    period_number,
    topic,
    allowed_pcs
):
    official_pc_details = (
        curriculum_data
        .get_performance_criteria(
            topic
        )
    )

    official_resources = (
        curriculum_data
        .get_resources(
            topic
        )
    )

    prompt = f"""
You are the CSC51 Weekly Planning Agent.

Create ONE complete 50-minute lesson.

COURSE:
{state["course"]}

GRADE:
{state["grade"]}

WEEK:
{state["week"]}

PERIOD:
{period_number}


OFFICIAL TOPIC:
{topic}

OFFICIAL PC IDS:
{allowed_pcs}

OFFICIAL PERFORMANCE CRITERIA:
{official_pc_details}

TRUSTED RESOURCES:
{official_resources}


ABSOLUTE RULES:

- Topic MUST remain exactly:
  "{topic}"

- selected_pcs MUST be exactly:
  {allowed_pcs}

- Do NOT invent another topic.

- Do NOT invent PC codes.

- Do NOT use PythonTutorial.pdf to choose the curriculum topic.

- Keep activities suitable for Grade 10 ADV.

- Include support, core and challenge differentiation.

- Include a real-life connection.

- Include 21st-century skills.

- Keep text concise and classroom-ready.

- Total lesson time must equal 50 minutes.


Return ONLY valid JSON:

{{
  "topic": "{topic}",

  "selected_pcs":
    {json.dumps(allowed_pcs)},

  "main_learning_focus":
    "...",

  "learning_objectives": [
    "...",
    "..."
  ],

  "success_criteria": [
    "...",
    "..."
  ],

  "real_life_connection":
    "...",

  "c21_skills":
    "...",

  "warm_up":
    "...",

  "focused_instruction":
    "...",

  "guided_practice":
    "...",

  "collaborative_learning":
    "...",

  "differentiation": {{
    "support":
      "...",

    "core":
      "...",

    "challenge":
      "..."
  }},

  "independent_learning":
    "...",

  "progress_check":
    "...",

  "home_learning":
    "...",

  "resources": [
    "..."
  ],

  "reflection_prompt":
    "...",

  "timings": {{
    "warm_up": 5,
    "focused_instruction": 10,
    "guided_practice": 10,
    "collaborative_learning": 10,
    "independent_learning": 10,
    "progress_check": 5
  }}
}}
"""

    response = (
        json_llm
        .invoke(
            prompt
        )
    )

    plan = (
        parse_agent_json(
            response.content,
            f"Planning Agent - Period {period_number}"
        )
    )

    plan[
        "topic"
    ] = topic

    plan[
        "selected_pcs"
    ] = allowed_pcs

    plan.setdefault(
        "differentiation",
        {}
    )

    plan[
        "differentiation"
    ].setdefault(
        "support",
        (
            "Provide scaffolded instructions, "
            "starter code and teacher guidance."
        )
    )

    plan[
        "differentiation"
    ].setdefault(
        "core",
        (
            "Complete the expected lesson task "
            "using the required Python skills."
        )
    )

    plan[
        "differentiation"
    ].setdefault(
        "challenge",
        (
            "Extend the task with a more complex "
            "Python problem."
        )
    )

    return plan


def planning_agent(
    state: TeachingState
):
    week_data = (
        curriculum_data
        .get_week_data(
            state[
                "week"
            ]
        )
    )

    topic_1 = (
        week_data[
            "period_1"
        ]
    )

    topic_2 = (
        week_data[
            "period_2"
        ]
    )

    pcs_1 = (
        curriculum_data
        .pc_ids_for_topic(
            topic_1
        )
    )

    pcs_2 = (
        curriculum_data
        .pc_ids_for_topic(
            topic_2
        )
    )

    print(
        "\nPlanning Period 1..."
    )

    period_1_plan = (
        plan_one_period(
            state=state,
            period_number=1,
            topic=topic_1,
            allowed_pcs=pcs_1
        )
    )

    print(
        "\nPlanning Period 2..."
    )

    period_2_plan = (
        plan_one_period(
            state=state,
            period_number=2,
            topic=topic_2,
            allowed_pcs=pcs_2
        )
    )

    plan = {
        "period_1":
            period_1_plan,

        "period_2":
            period_2_plan,
    }

    print(
        "\nPlanning Agent finished."
    )

    print(
        "Period 1:",
        topic_1,
        pcs_1
    )

    print(
        "Period 2:",
        topic_2,
        pcs_2
    )

    return {
        "weekly_lesson_plan":
            plan
    }


# ============================================================
# 3. DETERMINISTIC GROUNDING SAFEGUARD
# ============================================================

def grounding_safeguard(
    state: TeachingState
):
    week_data = (
        curriculum_data
        .get_week_data(
            state[
                "week"
            ]
        )
    )

    topic_1 = (
        week_data[
            "period_1"
        ]
    )

    topic_2 = (
        week_data[
            "period_2"
        ]
    )

    allowed_1 = set(
        curriculum_data
        .pc_ids_for_topic(
            topic_1
        )
    )

    allowed_2 = set(
        curriculum_data
        .pc_ids_for_topic(
            topic_2
        )
    )

    plan = (
        state[
            "weekly_lesson_plan"
        ]
    )

    period_1_pcs = set(
        plan[
            "period_1"
        ].get(
            "selected_pcs",
            []
        )
    )

    period_2_pcs = set(
        plan[
            "period_2"
        ].get(
            "selected_pcs",
            []
        )
    )

    topic_1_correct = (
        plan[
            "period_1"
        ].get(
            "topic"
        )
        == topic_1
    )

    topic_2_correct = (
        plan[
            "period_2"
        ].get(
            "topic"
        )
        == topic_2
    )

    passed = (
        period_1_pcs
        == allowed_1

        and period_2_pcs
        == allowed_2

        and topic_1_correct

        and topic_2_correct
    )

    result = {
        "passed":
            passed,

        "period_1_topic":
            plan[
                "period_1"
            ].get(
                "topic"
            ),

        "period_1_expected_topic":
            topic_1,

        "period_1_pcs":
            sorted(
                period_1_pcs
            ),

        "period_1_allowed_pcs":
            sorted(
                allowed_1
            ),

        "period_2_topic":
            plan[
                "period_2"
            ].get(
                "topic"
            ),

        "period_2_expected_topic":
            topic_2,

        "period_2_pcs":
            sorted(
                period_2_pcs
            ),

        "period_2_allowed_pcs":
            sorted(
                allowed_2
            ),
    }

    if passed:
        print(
            "\nGrounding safeguard: PASSED"
        )

    else:
        print(
            "\nGrounding safeguard: FAILED"
        )

        print(
            json.dumps(
                result,
                indent=2
            )
        )

    return {
        "grounding_check":
            result
    }


# ============================================================
# 4. RICH RESOURCE AGENT
# ============================================================

def resource_requirements_for_pcs(
    selected_pcs
):
    selected = set(
        selected_pcs
    )

    requirements = []

    if "PC1.1" in selected:
        requirements.append(
            """
PC1.1 MUST be explicitly taught:
- explain what a nested list is
- create a nested list
- access an element inside a nested list
"""
        )

    if "PC1.2" in selected:
        requirements.append(
            """
PC1.2 MUST be explicitly taught and assessed:
- positive indexing
- negative indexing
- changing a list item using assignment
- append()
- insert()

At least TWO Period 1 slides must visibly support PC1.2.
The progress check must contain at least one PC1.2 question.
The worksheet questions for this period must include at least one
question using indexing, append(), or insert().
"""
        )

    if "PC1.3" in selected:
        requirements.append(
            """
PC1.3 MUST be explicitly taught:
- Bubble Sort as an algorithm
- compare adjacent values
- swap values when needed
- repeat passes until sorted
- Python sort()
- reverse()
- len()

Clearly distinguish Bubble Sort from Python's built-in sort().
"""
        )

    if not requirements:
        requirements.append(
            """
Follow every selected PC exactly and ensure each selected PC
appears explicitly in at least one teaching slide and one assessment item.
"""
        )

    return "\n".join(
        requirements
    )


def generate_period_resources(
    state,
    period_key,
    period_number
):
    lesson = (
        state[
            "weekly_lesson_plan"
        ][
            period_key
        ]
    )

    topic = (
        lesson[
            "topic"
        ]
    )

    selected_pcs = (
        lesson[
            "selected_pcs"
        ]
    )

    tutorial_material = (
        curriculum_data
        .get_tutorial_material(
            topic
        )
    )

    official_pc_details = (
        curriculum_data
        .get_performance_criteria(
            topic
        )
    )

    official_resources = (
        curriculum_data
        .get_resources(
            topic
        )
    )

    required_pc_content = (
        resource_requirements_for_pcs(
            selected_pcs
        )
    )

    prompt = f"""
You are the CSC51 Teaching Resource Agent.

Create teaching resources for ONE lesson period only.

================================================
APPROVED LESSON
================================================

Period:
{period_number}

Topic:
{topic}

Selected PCs:
{selected_pcs}

Official PC details:
{official_pc_details}

Approved lesson plan:
{json.dumps(lesson, indent=2)}


================================================
REQUIRED PC COVERAGE
================================================

{required_pc_content}


================================================
TRUSTED RESOURCES
================================================

{official_resources}


================================================
RELEVANT PythonTutorial.pdf MATERIAL
================================================

{tutorial_material}


ABSOLUTE RULES:

1. The approved lesson plan and official PCs are the authority.

2. PythonTutorial.pdf is ONLY a supporting source for
   explanations, examples, code and visuals.

3. Ignore tutorial content that does not directly support
   the selected PCs.

4. Do NOT change the topic.

5. Do NOT change the PCs.

6. Do NOT introduce unrelated Python topics.

7. EVERY selected PC must appear explicitly in:
   - the Lesson Focus slide
   - at least one teaching slide
   - the Progress Check

8. The worksheet questions must collectively assess
   ALL selected PCs.

9. Explanations must be suitable for Grade 10 ADV.

10. Code examples must be short, meaningful and classroom-ready.


================================================
PERIOD 1 WEEK 1 STRUCTURE
================================================

If the selected PCs are PC1.1 and PC1.2,
the seven slides MUST follow this structure:

Slide 1:
Lesson Focus
- show PC1.1 AND PC1.2
- show learning objectives

Slide 2:
Nested Lists
- PC1.1
- definition and simple example

Slide 3:
Positive and Negative Indexing
- PC1.2
- show both positive and negative index positions
- include a code example

Slide 4:
Visual Indexing Explanation
- PC1.2
- use cards/process/comparison
- show how index 0 and index -1 locate elements

Slide 5:
Changing Lists
- PC1.2
- demonstrate assignment
- demonstrate append()
- demonstrate insert()
- include code

Slide 6:
Guided and Differentiated Practice
- include PC1.1 and PC1.2
- support, core and challenge

Slide 7:
Progress Check
- include at least one PC1.1 question
- include at least two PC1.2 questions


================================================
PERIOD 2 WEEK 1 STRUCTURE
================================================

If PC1.3 is selected,
the seven slides MUST follow this structure:

Slide 1:
Lesson Focus
- PC1.3
- learning objectives

Slide 2:
Bubble Sort Concept
- compare adjacent values
- swap if needed
- repeat

Slide 3:
Bubble Sort Worked Example
- show a short list
- explain one or more passes

Slide 4:
Bubble Sort Visual Process
- process diagram
- compare, swap, move, repeat

Slide 5:
Python Built-in List Utilities
- sort()
- reverse()
- len()
- explain that these are different from implementing Bubble Sort

Slide 6:
Guided and Differentiated Practice

Slide 7:
Progress Check
- Bubble Sort question
- sort()/reverse()/len() question


================================================
DO NOT TEACH UNLESS OFFICIALLY REQUIRED
================================================

Do NOT introduce these unrelated topics:

- tuples
- dictionaries
- stack operations
- pop()
- count()
- copy()
- zip()
- enumerate()
- dictionary looping
- unrelated looping techniques


================================================
POWERPOINT REQUIREMENTS
================================================

Create EXACTLY 7 slides for this period.

For each slide provide:
- title
- slide_type
- bullets
- code
- visual_type
- visual_items
- source_page
- supports_pc

Allowed visual_type values:

"cards"
"process"
"comparison"
"source_page"
"none"

Use a real PAGE number from PythonTutorial.pdf
when the retrieved material contains a relevant page.
Otherwise use null.


================================================
WORKSHEET
================================================

Create exactly 3 questions for this period.

The three questions must collectively assess
EVERY selected PC for this period.


Return ONLY valid JSON:

{{
  "slides": [

    {{
      "title":
        "...",

      "slide_type":
        "concept",

      "bullets": [
        "...",
        "..."
      ],

      "code":
        "",

      "visual_type":
        "cards",

      "visual_items": [
        "...",
        "...",
        "..."
      ],

      "source_page":
        null,

      "supports_pc": [
        "PC..."
      ]
    }}

  ],

  "worksheet_questions": [

    {{
      "number": 1,
      "question": "...",
      "type": "recall"
    }},

    {{
      "number": 2,
      "question": "...",
      "type": "write_code"
    }},

    {{
      "number": 3,
      "question": "...",
      "type": "challenge"
    }}

  ]
}}
"""

    response = (
        resource_llm
        .invoke(
            prompt
        )
    )

    data = (
        parse_agent_json(
            response.content,
            f"Resource Agent - Period {period_number}"
        )
    )

    slides = (
        data.get(
            "slides",
            []
        )
    )

    questions = (
        data.get(
            "worksheet_questions",
            []
        )
    )

    if len(
        slides
    ) != 7:
        raise ValueError(
            f"Period {period_number} must generate exactly 7 slides. "
            f"Generated: {len(slides)}"
        )

    if len(
        questions
    ) != 3:
        raise ValueError(
            f"Period {period_number} must generate exactly 3 "
            f"worksheet questions. Generated: {len(questions)}"
        )

    allowed_pcs = set(
        selected_pcs
    )

    for slide in slides:
        slide_pcs = set(
            slide.get(
                "supports_pc",
                []
            )
        )

        unsupported = (
            slide_pcs
            - allowed_pcs
        )

        if unsupported:
            raise ValueError(
                "Resource Agent introduced unsupported PCs: "
                f"{sorted(unsupported)}"
            )

    # --------------------------------------------------------
    # DETERMINISTIC CONTENT VALIDATION
    # --------------------------------------------------------

    resource_text = (
        json.dumps(
            {
                "slides":
                    slides,

                "questions":
                    questions,
            }
        )
        .lower()
    )

    missing = []

    if "PC1.1" in allowed_pcs:
        if "nested" not in resource_text:
            missing.append(
                "PC1.1 nested lists"
            )

    if "PC1.2" in allowed_pcs:
        required_pc12_terms = [
            "positive",
            "negative",
            "append",
            "insert",
        ]

        for term in required_pc12_terms:
            if term not in resource_text:
                missing.append(
                    f"PC1.2 required concept: {term}"
                )

    if "PC1.3" in allowed_pcs:
        required_pc13_terms = [
            "bubble sort",
            "swap",
            "sort()",
            "reverse()",
            "len()",
        ]

        for term in required_pc13_terms:
            if term not in resource_text:
                missing.append(
                    f"PC1.3 required concept: {term}"
                )

    if missing:
        raise ValueError(
            "Resource Agent missed required curriculum content:\n- "
            + "\n- ".join(
                missing
            )
        )

    return (
        slides,
        questions
    )


def resource_agent(
    state: TeachingState
):
    print(
        "\nCreating Period 1 resources..."
    )

    slides_1, questions_1 = (
        generate_period_resources(
            state=state,
            period_key="period_1",
            period_number=1
        )
    )

    print(
        "\nCreating Period 2 resources..."
    )

    slides_2, questions_2 = (
        generate_period_resources(
            state=state,
            period_key="period_2",
            period_number=2
        )
    )

    slides = (
        slides_1
        + slides_2
    )

    combined_questions = (
        questions_1
        + questions_2
    )

    all_questions = []

    for index, question in enumerate(
        combined_questions,
        start=1
    ):
        question[
            "number"
        ] = index

        all_questions.append(
            question
        )

    powerpoint = {
        "slides":
            slides
    }

    worksheet = {
        "title":
            (
                f"CSC51 Week "
                f"{state['week']} Worksheet"
            ),

        "instructions":
            (
                "Complete all questions. "
                "Use Python code where required."
            ),

        "questions":
            all_questions
    }

    print(
        "\nResource Agent finished."
    )

    print(
        "Period 1 slides:",
        len(
            slides_1
        )
    )

    print(
        "Period 2 slides:",
        len(
            slides_2
        )
    )

    print(
        "Total content slides:",
        len(
            slides
        )
    )

    print(
        "Worksheet questions:",
        len(
            all_questions
        )
    )

    return {
        "powerpoint_content":
            powerpoint,

        "worksheet":
            worksheet,
    }


# ============================================================
# 5. REVIEWER AGENT / LLM-AS-A-JUDGE
# ============================================================

def reviewer_agent(
    state: TeachingState
):
    slides = (
        state
        .get(
            "powerpoint_content",
            {}
        )
        .get(
            "slides",
            []
        )
    )

    questions = (
        state
        .get(
            "worksheet",
            {}
        )
        .get(
            "questions",
            []
        )
    )

    visual_count = sum(
        1
        for slide in slides
        if str(
            slide.get(
                "visual_type",
                "none"
            )
        ).lower()
        != "none"
    )

    code_count = sum(
        1
        for slide in slides
        if str(
            slide.get(
                "code",
                ""
            )
        ).strip()
    )

    source_page_count = sum(
        1
        for slide in slides
        if slide.get(
            "source_page"
        )
        not in (
            None,
            "",
            0
        )
    )

    deterministic_quality = {
        "slide_count":
            len(
                slides
            ),

        "visual_slide_count":
            visual_count,

        "code_slide_count":
            code_count,

        "tutorial_source_slide_count":
            source_page_count,

        "worksheet_question_count":
            len(
                questions
            ),
    }

    resource_text = (
        json.dumps(
            {
                "slides":
                    slides,

                "questions":
                    questions,
            }
        )
        .lower()
    )

    deterministic_pc_coverage = {
        "pc1_1_nested_lists":
            (
                "nested"
                in resource_text
            ),

        "pc1_2_positive_indexing":
            (
                "positive"
                in resource_text
            ),

        "pc1_2_negative_indexing":
            (
                "negative"
                in resource_text
            ),

        "pc1_2_append":
            (
                "append"
                in resource_text
            ),

        "pc1_2_insert":
            (
                "insert"
                in resource_text
            ),

        "pc1_3_bubble_sort":
            (
                "bubble sort"
                in resource_text
            ),

        "pc1_3_sort":
            (
                "sort()"
                in resource_text
            ),

        "pc1_3_reverse":
            (
                "reverse()"
                in resource_text
            ),

        "pc1_3_len":
            (
                "len()"
                in resource_text
            ),
    }

    prompt = f"""
You are the final CSC51 Reviewer Agent.

Evaluate the generated teaching pack.

Do NOT automatically give high scores.

Check whether EVERY selected performance criterion is
actually taught and assessed.

Evaluate from 1 to 5 for:

1. curriculum alignment
2. completeness
3. clarity
4. retrieval grounding
5. PowerPoint teaching quality
6. visual richness
7. safety and age appropriateness


IMPORTANT:

If grounding_check.passed is false,
approved MUST be false.

For Week 1:
- PC1.1 requires nested-list coverage.
- PC1.2 must include positive indexing,
  negative indexing, append() and insert().
- PC1.3 must include Bubble Sort plus
  sort(), reverse() and len().


OFFICIAL CURRICULUM:

{state["curriculum_context"]}


LESSON PLAN:

{json.dumps(
    state["weekly_lesson_plan"],
    indent=2
)}


GROUNDING CHECK:

{json.dumps(
    state["grounding_check"],
    indent=2
)}


RESOURCE QUALITY COUNTS:

{json.dumps(
    deterministic_quality,
    indent=2
)}


DETERMINISTIC PC COVERAGE:

{json.dumps(
    deterministic_pc_coverage,
    indent=2
)}


POWERPOINT:

{json.dumps(
    state["powerpoint_content"],
    indent=2
)}


WORKSHEET:

{json.dumps(
    state["worksheet"],
    indent=2
)}


Return ONLY valid JSON:

{{
  "approved": true,

  "curriculum_alignment": 5,

  "completeness": 5,

  "clarity": 5,

  "retrieval_grounding": 5,

  "powerpoint_quality": 5,

  "visual_richness": 5,

  "safety": 5,

  "issues": [],

  "overall_comment":
    "..."
}}
"""

    response = (
        json_llm
        .invoke(
            prompt
        )
    )

    review = (
        parse_agent_json(
            response.content,
            "Reviewer Agent"
        )
    )

    review[
        "resource_quality_metrics"
    ] = (
        deterministic_quality
    )

    review[
        "deterministic_pc_coverage"
    ] = (
        deterministic_pc_coverage
    )

    hard_issues = []

    if not state[
        "grounding_check"
    ][
        "passed"
    ]:
        hard_issues.append(
            "Deterministic grounding safeguard failed."
        )

    if len(
        slides
    ) != 14:
        hard_issues.append(
            "PowerPoint must contain exactly 14 content slides."
        )

    if visual_count < 4:
        hard_issues.append(
            "PowerPoint has fewer than 4 visual slides."
        )

    if code_count < 4:
        hard_issues.append(
            "PowerPoint has fewer than 4 worked code examples."
        )

    if source_page_count < 2:
        hard_issues.append(
            "PowerPoint has fewer than 2 PythonTutorial.pdf "
            "source-page references."
        )

    if len(
        questions
    ) != 6:
        hard_issues.append(
            "Worksheet must contain exactly 6 questions."
        )

    # Week 1 curriculum coverage check
    if state[
        "week"
    ] == 1:
        required_week1_checks = {
            "PC1.1 nested lists":
                deterministic_pc_coverage[
                    "pc1_1_nested_lists"
                ],

            "PC1.2 positive indexing":
                deterministic_pc_coverage[
                    "pc1_2_positive_indexing"
                ],

            "PC1.2 negative indexing":
                deterministic_pc_coverage[
                    "pc1_2_negative_indexing"
                ],

            "PC1.2 append()":
                deterministic_pc_coverage[
                    "pc1_2_append"
                ],

            "PC1.2 insert()":
                deterministic_pc_coverage[
                    "pc1_2_insert"
                ],

            "PC1.3 Bubble Sort":
                deterministic_pc_coverage[
                    "pc1_3_bubble_sort"
                ],

            "PC1.3 sort()":
                deterministic_pc_coverage[
                    "pc1_3_sort"
                ],

            "PC1.3 reverse()":
                deterministic_pc_coverage[
                    "pc1_3_reverse"
                ],

            "PC1.3 len()":
                deterministic_pc_coverage[
                    "pc1_3_len"
                ],
        }

        for label, present in required_week1_checks.items():
            if not present:
                hard_issues.append(
                    f"Missing required Week 1 coverage: {label}"
                )

    if hard_issues:
        review[
            "approved"
        ] = False

        review.setdefault(
            "issues",
            []
        )

        review[
            "issues"
        ].extend(
            hard_issues
        )

    print(
        "\nReviewer Agent finished."
    )

    print(
        "Approved:",
        review.get(
            "approved"
        )
    )

    print(
        "Quality metrics:",
        deterministic_quality
    )

    print(
        "PC coverage:",
        deterministic_pc_coverage
    )

    if hard_issues:
        print(
            "Detected issues:"
        )

        for issue in hard_issues:
            print(
                "-",
                issue
            )

    return {
        "review":
            review
    }


print(
    "Updated specialized agents ready."
)


## 9. LangGraph handoff workflow


In [ ]:
def build_app():
    graph = StateGraph(TeachingState)

    graph.add_node("retrieval_agent", retrieval_node)
    graph.add_node("planning_agent", planning_agent)
    graph.add_node("grounding_safeguard", grounding_safeguard)
    graph.add_node("resource_agent", resource_agent)
    graph.add_node("reviewer_agent", reviewer_agent)

    graph.add_edge(START, "retrieval_agent")
    graph.add_edge("retrieval_agent", "planning_agent")
    graph.add_edge("planning_agent", "grounding_safeguard")
    graph.add_edge("grounding_safeguard", "resource_agent")
    graph.add_edge("resource_agent", "reviewer_agent")
    graph.add_edge("reviewer_agent", END)

    return graph.compile()

app = build_app()

print("LangGraph ready.")


## 10. Run the complete system

Choose the week in the dropdown above, then run this cell.

The selected week is read **directly from the dropdown**, so there is no stale `WEEK` variable.


In [ ]:
selected_week = int(
    week_selector.value
)

if not 1 <= selected_week <= 11:
    raise ValueError(
        "Week must be between 1 and 11."
    )

initial_state: TeachingState = {
    "week": selected_week,
    "grade": GRADE,
    "course": COURSE,
    "weekly_topics": "",
    "performance_criteria": "",
    "resources": "",
    "tutorial_material": "",
    "curriculum_context": "",
    "weekly_lesson_plan": {},
    "grounding_check": {},
    "powerpoint_content": {},
    "worksheet": {},
    "review": {},
    "output_files": [],
}

print(
    f"Generating Week {selected_week}..."
)

final_state = await app.ainvoke(
    initial_state
)

print(
    "\n MULTI-AGENT SYSTEM FINISHED"
)
print(
    "=" * 60
)

print(
    "\nWEEKLY TOPICS:"
)
print(
    final_state[
        "weekly_topics"
    ]
)

print(
    "\nSELECTED PCs:"
)
print(
    "Period 1:",
    final_state[
        "weekly_lesson_plan"
    ][
        "period_1"
    ][
        "selected_pcs"
    ]
)
print(
    "Period 2:",
    final_state[
        "weekly_lesson_plan"
    ][
        "period_2"
    ][
        "selected_pcs"
    ]
)

print(
    "\nPOWERPOINT:"
)
print(
    "Slides:",
    len(
        final_state[
            "powerpoint_content"
        ].get(
            "slides",
            []
        )
    )
)

print(
    "\nGROUNDING:"
)
print(
    json.dumps(
        final_state[
            "grounding_check"
        ],
        indent=2
    )
)

print(
    "\nFINAL REVIEW:"
)
print(
    json.dumps(
        final_state[
            "review"
        ],
        indent=2
    )
)

## 11. Inspect the generated weekly lesson plan


In [ ]:
print(json.dumps(
    final_state["weekly_lesson_plan"],
    indent=2
))


## 12. Rich artifact-generation tools

This version:
- fills the school Word template,
- creates a detailed 16:9 PowerPoint,
- uses grounded content retrieved from `PythonTutorial.pdf`,
- can render retrieved tutorial pages as actual slide visuals,
- adds code boxes and generated diagrams,
- validates slide richness before export.

In [ ]:
# ============================================================
# STEP 12 - FINAL ARTIFACT GENERATION TOOLS
# ============================================================

import pymupdf


# ============================================================
# GENERAL HELPERS
# ============================================================

def text_value(value):
    if isinstance(value, list):
        return "\n".join(
            f"• {x}"
            for x in value
        )

    if isinstance(value, dict):
        return "\n".join(
            f"{k}: {v}"
            for k, v in value.items()
        )

    return str(
        value or ""
    )


def normalize_text(text):
    return " ".join(
        str(
            text or ""
        )
        .lower()
        .replace(
            "\n",
            " "
        )
        .split()
    )


# ============================================================
# WORD TEMPLATE HELPERS
# ============================================================

def all_tables(document):
    def walk(table):
        yield table

        for row in table.rows:
            for cell in row.cells:
                for nested in cell.tables:
                    yield from walk(
                        nested
                    )

    for table in document.tables:
        yield from walk(
            table
        )


def unique_cells(row):
    result = []
    seen = set()

    for cell in row.cells:
        cell_id = id(
            cell._tc
        )

        if cell_id not in seen:
            seen.add(
                cell_id
            )

            result.append(
                cell
            )

    return result


def find_cell(
    document,
    search_text,
    occurrence=0
):
    target = normalize_text(
        search_text
    )

    matches = []

    for table in all_tables(
        document
    ):
        for row in table.rows:
            for cell in unique_cells(
                row
            ):
                current = normalize_text(
                    cell.text
                )

                if target in current:
                    matches.append(
                        (
                            table,
                            row,
                            cell
                        )
                    )

    if occurrence < len(
        matches
    ):
        return matches[
            occurrence
        ]

    return None


def replace_matching_cell(
    document,
    search_text,
    value,
    occurrence=0
):
    result = find_cell(
        document,
        search_text,
        occurrence
    )

    if result is None:
        print(
            "Warning: Could not find template text:",
            search_text
        )

        return False

    _, _, cell = result

    cell.text = text_value(
        value
    )

    return True


def append_to_matching_cell(
    document,
    search_text,
    value,
    occurrence=0
):
    result = find_cell(
        document,
        search_text,
        occurrence
    )

    if result is None:
        print(
            "Warning: Could not find template heading:",
            search_text
        )

        return False

    _, _, cell = result

    value = text_value(
        value
    )

    if value:
        cell.add_paragraph(
            value
        )

    return True


def set_value_after_label(
    document,
    label_text,
    value,
    occurrence=0
):
    result = find_cell(
        document,
        label_text,
        occurrence
    )

    if result is None:
        print(
            "Warning: Could not find label:",
            label_text
        )

        return False

    _, row, label_cell = result

    cells = unique_cells(
        row
    )

    try:
        index = cells.index(
            label_cell
        )

    except ValueError:
        return False

    if index + 1 < len(
        cells
    ):
        cells[
            index + 1
        ].text = text_value(
            value
        )

        return True

    return False


def set_stage_time(
    document,
    stage_text,
    minutes
):
    result = find_cell(
        document,
        stage_text
    )

    if result is None:
        return False

    _, row, stage_cell = result

    cells = unique_cells(
        row
    )

    try:
        index = cells.index(
            stage_cell
        )

    except ValueError:
        return False

    if index > 0:
        cells[
            index - 1
        ].text = (
            f"{minutes} min"
        )

        return True

    return False


# ============================================================
# SAVE LESSON PLAN
# ============================================================

def save_lesson_docx(
    state,
    period_key
):
    lesson = (
        state[
            "weekly_lesson_plan"
        ][
            period_key
        ]
    )

    period_number = (
        1
        if period_key == "period_1"
        else 2
    )

    doc = Document(
        LESSON_TEMPLATE_PATH
    )

    # Header
    set_value_after_label(
        doc,
        "CLASS",
        state["grade"]
    )

    set_value_after_label(
        doc,
        "WEEK & DATE",
        (
            f"Week {state['week']} "
            f"- Period {period_number}"
        )
    )

    # Topic
    append_to_matching_cell(
        doc,
        "CHAPTER",
        lesson.get(
            "topic",
            ""
        )
    )

    selected_pcs = text_value(
        lesson.get(
            "selected_pcs",
            []
        )
    )

    objectives = text_value(
        lesson.get(
            "learning_objectives",
            []
        )
    )

    learning_focus = (
        lesson.get(
            "main_learning_focus",
            ""
        )
        + "\n\nPerformance Criteria:\n"
        + selected_pcs
        + "\n\nLearning Objectives:\n"
        + objectives
    )

    append_to_matching_cell(
        doc,
        "MAIN LEARNING FOCUS",
        learning_focus
    )

    append_to_matching_cell(
        doc,
        "SUCCESS CRITERIA",
        lesson.get(
            "success_criteria",
            []
        )
    )

    c21_text = (
        "Real-life connection:\n"
        + lesson.get(
            "real_life_connection",
            ""
        )
        + "\n\n21st Century Skills:\n"
        + lesson.get(
            "c21_skills",
            ""
        )
    )

    append_to_matching_cell(
        doc,
        "C21ST SKILLS",
        c21_text
    )

    numeracy_text = (
        "Numeracy:\n"
        "Use indices, sequences, ordering and logical reasoning.\n\n"
        "Literacy:\n"
        "Explain Python code using correct technical vocabulary.\n\n"
        "Culture & Heritage:\n"
        "Use UAE-related examples or datasets where appropriate."
    )

    append_to_matching_cell(
        doc,
        "NUMERACY",
        numeracy_text
    )

    append_to_matching_cell(
        doc,
        "RESOURCES",
        lesson.get(
            "resources",
            []
        )
    )

    # Warm up
    set_stage_time(
        doc,
        "WARM UP",
        5
    )

    replace_matching_cell(
        doc,
        "Plan activities like brainstorming",
        lesson.get(
            "warm_up",
            ""
        )
    )

    # Focused instruction.
    # After the first matching cell is replaced, the remaining matching
    # cell becomes occurrence 0. This avoids the old occurrence=1 bug.
    set_stage_time(
        doc,
        "FOCUSED",
        10
    )

    replace_matching_cell(
        doc,
        "Detail your teaching activities",
        lesson.get(
            "focused_instruction",
            ""
        ),
        occurrence=0
    )

    # Guided practice
    set_stage_time(
        doc,
        "GUIDED",
        10
    )

    replace_matching_cell(
        doc,
        "Detail your teaching activities",
        lesson.get(
            "guided_practice",
            ""
        ),
        occurrence=0
    )

    # Collaborative / differentiation
    differentiation = lesson.get(
        "differentiation",
        {}
    )

    group_task = f"""
COLLABORATIVE TASK

{lesson.get("collaborative_learning", "")}

SUPPORT - NEED HELP
{differentiation.get("support", "")}

CORE - EXPECTED LEVEL
{differentiation.get("core", "")}

CHALLENGE - HIGH ACHIEVER
{differentiation.get("challenge", "")}
""".strip()

    set_stage_time(
        doc,
        "COLLABORAT",
        10
    )

    replace_matching_cell(
        doc,
        "Outline group activities",
        group_task
    )

    # Independent learning
    set_stage_time(
        doc,
        "INDEPENDENT",
        10
    )

    independent_text = f"""
INDIVIDUAL TASK

{lesson.get("independent_learning", "")}

Success evidence:
Students independently demonstrate the selected performance criteria.
""".strip()

    replace_matching_cell(
        doc,
        "Describe tasks that students will do independently",
        independent_text
    )

    # Progress check
    set_stage_time(
        doc,
        "PROGRESS",
        5
    )

    replace_matching_cell(
        doc,
        "Plan how you will assess",
        lesson.get(
            "progress_check",
            ""
        )
    )

    # Home learning
    home_result = find_cell(
        doc,
        "Home Learning"
    )

    if home_result is not None:
        _, _, cell = home_result

        cell.text = (
            "HOME LEARNING\n\n"
            + lesson.get(
                "home_learning",
                ""
            )
        )

    # Reflection
    reflection_result = find_cell(
        doc,
        "REFLECTION"
    )

    if reflection_result is not None:
        _, _, cell = reflection_result

        cell.text = (
            "REFLECTION "
            "(at the end of the lesson/unit/topic)"
            "\n\n"
            + lesson.get(
                "reflection_prompt",
                ""
            )
        )

    path = (
        OUTPUT_DIR
        / (
            f"Week_{state['week']}"
            f"_Period_{period_number}"
            "_Lesson_Plan.docx"
        )
    )

    doc.save(
        path
    )

    print(
        " Lesson plan saved:",
        path
    )

    return str(
        path
    )


# ============================================================
# POWERPOINT VISUAL HELPERS
# ============================================================

PPT_NAVY = RGBColor(
    28,
    42,
    74
)

PPT_BLUE = RGBColor(
    44,
    95,
    160
)

PPT_LIGHT = RGBColor(
    242,
    246,
    252
)

PPT_WHITE = RGBColor(
    255,
    255,
    255
)

PPT_DARK = RGBColor(
    30,
    30,
    30
)

PPT_CODE = RGBColor(
    30,
    34,
    45
)


def set_shape_text(
    shape,
    text,
    font_size=20,
    bold=False,
    color=None,
    align=None
):
    tf = shape.text_frame
    tf.clear()
    tf.word_wrap = True
    tf.vertical_anchor = (
        MSO_ANCHOR.MIDDLE
    )

    p = tf.paragraphs[0]
    p.text = str(
        text
    )

    if align is not None:
        p.alignment = align

    for run in p.runs:
        run.font.size = Pt(
            font_size
        )
        run.font.bold = bold

        if color is not None:
            run.font.color.rgb = (
                color
            )


def add_slide_title(
    slide,
    title
):
    box = slide.shapes.add_textbox(
        Inches(0.55),
        Inches(0.25),
        Inches(12.2),
        Inches(0.65)
    )

    tf = box.text_frame
    tf.clear()

    p = tf.paragraphs[0]
    p.text = str(
        title
    )

    for run in p.runs:
        run.font.size = Pt(
            28
        )
        run.font.bold = True
        run.font.color.rgb = (
            PPT_NAVY
        )


def add_footer(
    slide,
    state,
    slide_number,
    source_page=None
):
    footer_text = (
        f"CSC51 • Week {state['week']} • "
        f"Slide {slide_number}"
    )

    if source_page:
        footer_text += (
            " • Source: PythonTutorial.pdf "
            f"p.{source_page}"
        )

    box = slide.shapes.add_textbox(
        Inches(0.55),
        Inches(7.05),
        Inches(12.0),
        Inches(0.25)
    )

    tf = box.text_frame
    tf.clear()

    p = tf.paragraphs[0]
    p.text = footer_text

    for run in p.runs:
        run.font.size = Pt(
            9
        )
        run.font.color.rgb = (
            RGBColor(
                90,
                90,
                90
            )
        )


def add_bullets(
    slide,
    bullets,
    left=0.65,
    top=1.25,
    width=6.1,
    height=5.4
):
    box = slide.shapes.add_textbox(
        Inches(left),
        Inches(top),
        Inches(width),
        Inches(height)
    )

    tf = box.text_frame
    tf.clear()
    tf.word_wrap = True

    for index, bullet in enumerate(
        bullets
    ):
        p = (
            tf.paragraphs[0]
            if index == 0
            else tf.add_paragraph()
        )

        p.text = str(
            bullet
        )

        p.font.size = Pt(
            20
        )

        p.space_after = Pt(
            10
        )

        p.level = 0

    return box


def add_code_box(
    slide,
    code,
    left=7.15,
    top=1.35,
    width=5.55,
    height=4.9
):
    shape = slide.shapes.add_shape(
        MSO_SHAPE.ROUNDED_RECTANGLE,
        Inches(left),
        Inches(top),
        Inches(width),
        Inches(height)
    )

    shape.fill.solid()
    shape.fill.fore_color.rgb = (
        PPT_CODE
    )

    shape.line.color.rgb = (
        PPT_CODE
    )

    tf = shape.text_frame
    tf.clear()
    tf.margin_left = Inches(
        0.25
    )
    tf.margin_right = Inches(
        0.25
    )
    tf.margin_top = Inches(
        0.2
    )
    tf.word_wrap = True

    p = tf.paragraphs[0]
    p.text = str(
        code
    )

    for run in p.runs:
        run.font.name = (
            "Consolas"
        )
        run.font.size = Pt(
            15
        )
        run.font.color.rgb = (
            PPT_WHITE
        )


def add_cards_visual(
    slide,
    items
):
    items = [
        str(x)
        for x in items
    ][:6]

    if not items:
        return

    left = 7.15
    top = 1.55
    width = 5.55

    card_h = min(
        0.75,
        4.8 / max(
            len(items),
            1
        )
    )

    gap = 0.12

    for i, item in enumerate(
        items
    ):
        shape = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(left),
            Inches(
                top
                + i * (
                    card_h + gap
                )
            ),
            Inches(width),
            Inches(card_h)
        )

        shape.fill.solid()
        shape.fill.fore_color.rgb = (
            PPT_LIGHT
        )

        shape.line.color.rgb = (
            PPT_BLUE
        )

        set_shape_text(
            shape,
            item,
            font_size=16,
            bold=(
                len(items) <= 4
            ),
            color=PPT_DARK
        )


def add_process_visual(
    slide,
    items
):
    items = [
        str(x)
        for x in items
    ][:5]

    if not items:
        return

    start_x = 7.15
    y = 2.1
    total_w = 5.55

    box_w = (
        total_w
        / len(items)
        - 0.12
    )

    for i, item in enumerate(
        items
    ):
        x = (
            start_x
            + i
            * (
                box_w
                + 0.12
            )
        )

        shape = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(x),
            Inches(y),
            Inches(box_w),
            Inches(1.35)
        )

        shape.fill.solid()
        shape.fill.fore_color.rgb = (
            PPT_LIGHT
        )

        shape.line.color.rgb = (
            PPT_BLUE
        )

        set_shape_text(
            shape,
            item,
            font_size=13,
            bold=True,
            color=PPT_DARK,
            align=PP_ALIGN.CENTER
        )

        if i < len(items) - 1:
            arrow = slide.shapes.add_shape(
                MSO_SHAPE.RIGHT_ARROW,
                Inches(
                    x
                    + box_w
                    - 0.05
                ),
                Inches(
                    y
                    + 0.45
                ),
                Inches(0.25),
                Inches(0.35)
            )

            arrow.fill.solid()
            arrow.fill.fore_color.rgb = (
                PPT_BLUE
            )

            arrow.line.color.rgb = (
                PPT_BLUE
            )


def add_comparison_visual(
    slide,
    items
):
    items = [
        str(x)
        for x in items
    ]

    if not items:
        return

    midpoint = max(
        1,
        len(items) // 2
    )

    left_items = (
        items[:midpoint]
    )

    right_items = (
        items[midpoint:]
    )

    if not right_items:
        right_items = [
            "Compare / discuss"
        ]

    for x, values in [
        (
            7.15,
            left_items
        ),
        (
            10.0,
            right_items
        )
    ]:
        shape = slide.shapes.add_shape(
            MSO_SHAPE.ROUNDED_RECTANGLE,
            Inches(x),
            Inches(1.7),
            Inches(2.55),
            Inches(3.7)
        )

        shape.fill.solid()
        shape.fill.fore_color.rgb = (
            PPT_LIGHT
        )

        shape.line.color.rgb = (
            PPT_BLUE
        )

        tf = shape.text_frame
        tf.clear()
        tf.word_wrap = True

        for i, item in enumerate(
            values
        ):
            p = (
                tf.paragraphs[0]
                if i == 0
                else tf.add_paragraph()
            )

            p.text = str(
                item
            )

            p.font.size = Pt(
                15
            )

            p.space_after = Pt(
                8
            )


def render_tutorial_page(
    page_number
):
    page_number = int(
        page_number
    )

    image_path = (
        IMAGE_CACHE_DIR
        / (
            f"PythonTutorial_page_"
            f"{page_number}.png"
        )
    )

    if image_path.exists():
        return image_path

    if not PYTHON_TUTORIAL_PATH.exists():
        return None

    pdf = pymupdf.open(
        str(
            PYTHON_TUTORIAL_PATH
        )
    )

    if (
        page_number < 1
        or page_number > len(pdf)
    ):
        pdf.close()
        return None

    page = pdf[
        page_number - 1
    ]

    pix = page.get_pixmap(
        matrix=pymupdf.Matrix(
            1.5,
            1.5
        ),
        alpha=False
    )

    pix.save(
        str(
            image_path
        )
    )

    pdf.close()

    return image_path


def add_source_page_visual(
    slide,
    page_number
):
    image_path = render_tutorial_page(
        page_number
    )

    if image_path is None:
        return False

    slide.shapes.add_picture(
        str(
            image_path
        ),
        Inches(7.35),
        Inches(1.2),
        width=Inches(5.0),
        height=Inches(5.55)
    )

    return True


# ============================================================
# SAVE RICH POWERPOINT
# ============================================================

def save_powerpoint(
    state
):
    prs = Presentation()

    prs.slide_width = Inches(
        13.333
    )

    prs.slide_height = Inches(
        7.5
    )

    # Title slide
    title_slide = prs.slides.add_slide(
        prs.slide_layouts[0]
    )

    title_slide.shapes.title.text = (
        f"CSC51 • Week {state['week']}"
    )

    subtitle = (
        title_slide
        .placeholders[1]
    )

    subtitle.text = (
        state[
            "weekly_topics"
        ]
        + "\nGrade 10 ADV"
        + "\nGrounded with PythonTutorial.pdf"
    )

    slides = (
        state[
            "powerpoint_content"
        ]
        .get(
            "slides",
            []
        )
    )

    for slide_number, item in enumerate(
        slides,
        start=2
    ):
        slide = prs.slides.add_slide(
            prs.slide_layouts[6]
        )

        title = item.get(
            "title",
            "CSC51"
        )

        add_slide_title(
            slide,
            title
        )

        bullets = item.get(
            "bullets",
            []
        )

        add_bullets(
            slide,
            bullets
        )

        code = str(
            item.get(
                "code",
                ""
            )
            or ""
        ).strip()

        visual_type = str(
            item.get(
                "visual_type",
                "none"
            )
            or "none"
        ).lower()

        visual_items = (
            item.get(
                "visual_items",
                []
            )
            or []
        )

        source_page = (
            item.get(
                "source_page"
            )
        )

        # Right-side visual priority:
        # source page > code > generated visual.
        visual_added = False

        if (
            visual_type
            == "source_page"
            and source_page
        ):
            visual_added = (
                add_source_page_visual(
                    slide,
                    source_page
                )
            )

        if (
            not visual_added
            and code
        ):
            add_code_box(
                slide,
                code
            )

            visual_added = True

        if not visual_added:
            if visual_type == "process":
                add_process_visual(
                    slide,
                    visual_items
                )

            elif visual_type == "comparison":
                add_comparison_visual(
                    slide,
                    visual_items
                )

            else:
                add_cards_visual(
                    slide,
                    visual_items
                )

        add_footer(
            slide,
            state,
            slide_number,
            source_page
        )

    path = (
        OUTPUT_DIR
        / (
            f"Week_{state['week']}"
            "_Teaching_Slides_RICH.pptx"
        )
    )

    prs.save(
        path
    )

    print(
        " Rich PowerPoint saved:",
        path
    )

    return str(
        path
    )


# ============================================================
# SAVE WORKSHEET
# ============================================================

def save_worksheet(
    state
):
    data = state[
        "worksheet"
    ]

    doc = Document()

    doc.add_heading(
        data.get(
            "title",
            f"CSC51 Week {state['week']} Worksheet"
        ),
        level=1
    )

    doc.add_paragraph(
        data.get(
            "instructions",
            (
                "Complete all questions. "
                "Use your class notes and "
                "code examples where needed."
            )
        )
    )

    doc.add_heading(
        "Student Details",
        level=2
    )

    doc.add_paragraph(
        "Name: __________________________"
    )

    doc.add_paragraph(
        "Class: __________________________"
    )

    doc.add_paragraph(
        "Date: ___________________________"
    )

    for question in data.get(
        "questions",
        []
    ):
        number = question.get(
            "number",
            ""
        )

        q_type = str(
            question.get(
                "type",
                ""
            )
        ).replace(
            "_",
            " "
        ).title()

        doc.add_heading(
            f"Question {number} - {q_type}",
            level=2
        )

        doc.add_paragraph(
            question.get(
                "question",
                ""
            )
        )

        doc.add_paragraph(
            "\nAnswer:\n"
            "________________________________________\n"
            "________________________________________\n"
            "________________________________________\n"
            "________________________________________"
        )

    path = (
        OUTPUT_DIR
        / (
            f"Week_{state['week']}"
            "_Worksheet.docx"
        )
    )

    doc.save(
        path
    )

    print(
        " Worksheet saved:",
        path
    )

    return str(
        path
    )


# ============================================================
# SAVE REVIEW
# ============================================================

def save_review(
    state
):
    path = (
        OUTPUT_DIR
        / (
            f"Week_{state['week']}"
            "_Review.json"
        )
    )

    path.write_text(
        json.dumps(
            state["review"],
            indent=2
        ),
        encoding="utf-8"
    )

    return str(
        path
    )


# ============================================================
# VALIDATE BEFORE EXPORT
# ============================================================

def validate_before_export(
    state
):
    slides = (
        state
        .get(
            "powerpoint_content",
            {}
        )
        .get(
            "slides",
            []
        )
    )

    questions = (
        state
        .get(
            "worksheet",
            {}
        )
        .get(
            "questions",
            []
        )
    )

    visual_count = sum(
        1
        for s in slides
        if str(
            s.get(
                "visual_type",
                "none"
            )
        ).lower() != "none"
    )

    code_count = sum(
        1
        for s in slides
        if str(
            s.get(
                "code",
                ""
            )
        ).strip()
    )

    source_count = sum(
        1
        for s in slides
        if s.get(
            "source_page"
        ) not in (
            None,
            "",
            0
        )
    )

    errors = []

    if not state.get(
        "weekly_lesson_plan"
    ):
        errors.append(
            "Lesson plan is empty."
        )

    if len(slides) < 12:
        errors.append(
            "PowerPoint has fewer than 12 content slides."
        )

    if visual_count < 5:
        errors.append(
            "PowerPoint needs at least 5 visual slides."
        )

    if code_count < 3:
        errors.append(
            "PowerPoint needs at least 3 worked code examples."
        )

    if source_count < 2:
        errors.append(
            "PowerPoint needs at least 2 PythonTutorial.pdf "
            "page references."
        )

    if len(questions) < 6:
        errors.append(
            "Worksheet needs at least 6 questions."
        )

    print(
        "\nEXPORT QUALITY CHECK"
    )
    print(
        "Slides:",
        len(slides)
    )
    print(
        "Visual slides:",
        visual_count
    )
    print(
        "Code slides:",
        code_count
    )
    print(
        "Tutorial-source slides:",
        source_count
    )
    print(
        "Worksheet questions:",
        len(questions)
    )

    if errors:
        raise ValueError(
            "\n".join(
                errors
            )
        )

    print(
        " Export validation passed."
    )


# ============================================================
# SAVE ALL OUTPUTS
# ============================================================

def save_all_outputs(
    state
):
    validate_before_export(
        state
    )

    files = [
        save_lesson_docx(
            state,
            "period_1"
        ),

        save_lesson_docx(
            state,
            "period_2"
        ),

        save_powerpoint(
            state
        ),

        save_worksheet(
            state
        ),

        save_review(
            state
        ),
    ]

    print(
        "\n ALL TEACHING PACK FILES SAVED"
    )

    for file_path in files:
        print(
            " -",
            file_path
        )

    return files


print(
    " Final rich artifact tools ready."
)

## 13. Save the teaching pack

Run this only after the multi-agent system has finished.


In [ ]:
if not final_state["review"].get("approved", False):
    print(
        "WARNING: Reviewer did not approve the pack. "
        "Files will still be saved for teacher review."
    )

try:
    generated_files = save_all_outputs(
        final_state
    )

except PermissionError as e:
    print(
        "\n A generated Word/PowerPoint file is currently open."
    )
    print(
        "Close the file in Microsoft Word/PowerPoint, then run Step 13 again."
    )
    raise

final_state["output_files"] = (
    generated_files
)

print(
    "\nGenerated files:"
)

for path in generated_files:
    print(
        "-",
        path
    )

## 14. Optional MCP verification from PowerShell

The interactive notebook uses direct LangChain function calling because Windows Jupyter `stdio` can raise a `fileno` error.

For your final capstone evidence, test the MCP server separately from PowerShell:

```powershell
cd "YOUR_PROJECT_FOLDER"
python mcp_server.py
```

The server should start without a Python exception.  
For the GitHub version, keep both `curriculum_data.py` and `mcp_server.py`.
